# ASR L2-Accent - Phase B analysis package (Colab)

Computes the **oracle head-room**, the **LM-strength ladder** (does the rescoring
gain scale with LM quality, and how close to the oracle ceiling?), and a
**genuinely fine-tuned in-domain LM** (audio-free domain adaptation).

All text-only on the cached n-best - no audio - so this is fast and the upload is
~2.5 MB. GPU is only needed for the big ladder rungs and the fine-tune.

**Before running:** Runtime -> Change runtime type -> T4 GPU.
You need `asr_l2_analysis_bundle.zip` (made locally by
`python scripts/make_analysis_colab.py`).

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q jiwer pandas pyarrow pyyaml tqdm transformers accelerate

In [ ]:
# Upload asr_l2_analysis_bundle.zip and unpack
import zipfile, os
from google.colab import files
up = files.upload()
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall('/content')
os.chdir('/content/asr-l2-accent')
print('cwd:', os.getcwd()); print(sorted(os.listdir()))

## 1. Oracle upper bound (instant, no GPU)
How much of the error is recoverable from the n-best by reranking alone.

In [ ]:
!python scripts/oracle_bound.py --dataset l2arctic_24spk

## 2. Fine-tune a small LM on the in-domain corpus (GPU)
Leakage-guarded (asserts zero overlap with the eval prompts). Turns the zero-shot
neural rescorer into a domain-adapted one.

In [ ]:
!python scripts/finetune_lm.py --base gpt2 --epochs 3

## 3. LM-strength ladder (GPU for the big rungs)
4-gram -> distilGPT-2 -> GPT-2 -> medium -> large -> xl (1.5B), plus the
fine-tuned in-domain GPT-2. Reports per-word perplexity (strength axis), test
WER delta, and fraction of the oracle head-room captured.

In [ ]:
neural_args = [
  "models/lm/l2arctic_4gram.pkl=4-gram (weak)",
  "neural:distilgpt2=distilGPT-2 (82M)",
  "neural:gpt2=GPT-2 (124M)",
  "neural:gpt2-medium=GPT-2-medium (355M)",
  "neural:gpt2-large=GPT-2-large (774M)",
  "neural:gpt2-xl=GPT-2-xl (1.5B)",
  "neural:models/lm/ft_gpt2_l2arctic=GPT-2 fine-tuned (in-domain)",
]
import subprocess, sys
cmd = [sys.executable, "scripts/lm_ladder.py", "--dataset", "l2arctic_24spk", "--lms", *neural_args]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Inspect the ladder
import json
d = json.load(open("results/l2arctic_24spk/phaseB_lm_ladder.json"))
print(f"baseline {d['test_wer_baseline']:.4f}  oracle {d['test_wer_oracle']:.4f}")
for r in d["ladder"]:
    cap = r["frac_headroom_captured"]
    print(f"  {r['label']:34s} ppl={r['ppl_word']:7.1f}  "
          f"WER->{r['test_wer_adapted']:.4f} ({r['delta_rel_pct']:+5.1f}%)  "
          f"headroom {None if cap is None else round(cap*100)}%")

## 4. Download results

In [ ]:
import zipfile, glob
out = "phaseB_analysis_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("results/l2arctic_24spk/phaseB_lm_ladder.json") + \
             glob.glob("results/l2arctic_24spk/phaseB_oracle.json"):
        z.write(f)
    # ship the fine-tuned LM too (small) so the ladder is reproducible offline
    for f in glob.glob("models/lm/ft_gpt2_l2arctic/**/*", recursive=True):
        z.write(f)
from google.colab import files
files.download(out)
print("downloaded", out)